# 20. Linked lists and trees

After this notebook you'll be able to:

- read LeetCode's `ListNode` and `TreeNode`,
- build linked lists and trees by hand, to test on your laptop,
- walk a linked list with `while node:`, without falling off the end,
- solve tree problems with recursion, where each node asks its children.

This is one of the harder notebooks. Go slowly, run every cell, and predict before you run when it asks you to.

## Part 1: linked lists

### A train

Think of a train. Each coach carries passengers, and each coach is coupled to the **next** coach. There's no list of coaches anywhere: to reach coach 3, you start at the engine and walk through coach 1, then coach 2. The last coach is coupled to nothing.

A **linked list** is a train of objects. Each object is a **node**, and each node holds two things:

- **`val`**: its value (the passengers),
- **`next`**: the next node (the coupling), or `None` if it's the last one.

The first node is called the **head**. You only ever get the head; everything else you reach by following `next`.

It's the same idea as `tanvi.best_friend.name` in notebook 19: an attribute that holds another object.

### LeetCode's definition

At the top of every linked list problem, LeetCode shows this:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
```

It's commented out because LeetCode already has the class behind the scenes. On your laptop the class has to really exist, so here it is without the `#`s. Run it (it prints nothing; it only defines the form).

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

A form with two boxes, `val` and `next`. Both have default values, so `ListNode()` makes a node with `val` 0 and `next` `None`.

### Building `1 -> 2 -> 3` by hand

Build the train from the back, so each node can be coupled to one that already exists.

In [ ]:
third = ListNode(3)
second = ListNode(2, third)
first = ListNode(1, second)
head = first

print(head.val)
print(head.next.val)
print(head.next.next.val)
print(head.next.next.next)

```
1
2
3
None
```

```
 head
  │
  ▼
 ┌─────┬──────┐    ┌─────┬──────┐    ┌─────┬──────┐
 │ val │ next │    │ val │ next │    │ val │ next │
 │  1  │  ●───┼───►│  2  │  ●───┼───►│  3  │ None │
 └─────┴──────┘    └─────┴──────┘    └─────┴──────┘

 head.val             -> 1
 head.next.val        -> 2
 head.next.next.val   -> 3
 head.next.next.next  -> None   (the end of the train)
```

Read `head.next.next.val` left to right, one dot at a time: start at `head`, go to its next node, go to that node's next node, read its value.

**Predict first:** what do these print? Write your guesses down, then run.

In [ ]:
print(second.val)
print(second.next.val)
print(third.next)

```
2
3
None
```

`second` is the middle coach, so its `next` is the coach holding 3. `third` is the last coach, coupled to nothing.

You can also build from the front, by setting `next` afterwards:

In [ ]:
h = ListNode(10)
h.next = ListNode(20)
h.next.next = ListNode(30)
print(h.val, h.next.val, h.next.next.val)

```
10 20 30
```

### Walking the list: `while node:`

You can't use `for` or `len` on a linked list. You walk it yourself, one coach at a time. Run this; the `print` inside the loop shows every step.

In [ ]:
node = head
while node:
    print("at the node holding", node.val)
    node = node.next
print("node is now", node)

```
at the node holding 1
at the node holding 2
at the node holding 3
node is now None
```

| Step | `node` is | prints | `node = node.next` makes `node` |
| --- | --- | --- | --- |
| 1 | the node holding 1 | `1` | the node holding 2 |
| 2 | the node holding 2 | `2` | the node holding 3 |
| 3 | the node holding 3 | `3` | `None` |
| check | `None` | | `while node:` is now `False`, the loop stops |

These two lines are the heart of every linked list problem:

- `while node:` means "while there **is** a node" (while `node` isn't `None`).
- `node = node.next` steps forward one coach.

Use a separate name like `node` for walking, so `head` still points at the start when you're done.

### Doing something on the walk

Walking is the same as a `for` loop over a list, only written by hand. All the loop patterns from notebook 09 work: count, sum, search. Here's a sum, in the LeetCode shape. `ListNode | None` is a type hint that means "a `ListNode`, or `None`": LeetCode's reminder that the list can be empty.

**Predict first:** what will it print for the list `1 -> 2 -> 3`? And for an empty list (`None`)?

In [ ]:
class Solution:
    def sumList(self, head: ListNode | None) -> int:
        total = 0
        node = head
        while node:
            total += node.val
            node = node.next
        return total

print(Solution().sumList(head))
print(Solution().sumList(None))

```
6
0
```

An empty list is `head = None`. The loop never runs, so the total stays 0. No crash.

### The error you'll see most: `'NoneType' object has no attribute 'val'`

Walk one step too far, and you're holding `None`, not a node. `None` has no `val`. Run this:

In [ ]:
node = head
while node:
    node = node.next
print(node.val)

```
AttributeError: 'NoneType' object has no attribute 'val'
```

Read it as: "you asked for `.val` on something that is `None`". **NoneType** is the type of `None`. After the loop, `node` is `None`, because that's what stopped the loop. The same message with `'next'`, `'left'` or `'right'` means the same thing.

Two ways to stay safe:

1. **Check first.** At the start of a method, `if not head: return ...` handles the empty list.
2. **Check before you look ahead.** If you need the next node, check that it exists: `while node and node.next:`. Python reads `and` left to right and stops at the first `False`, so when `node` is `None` it never touches `node.next`.

In [ ]:
node = head
while node and node.next:
    print(node.val, "is followed by", node.next.val)
    node = node.next
print("stopped at", node.val, "because it has no next")

```
1 is followed by 2
2 is followed by 3
stopped at 3 because it has no next
```

This loop stops **on** the last node instead of after it. That's how you find the last value.

### Helpers to test on your laptop

LeetCode writes a linked list as `head = [1,2,3]` and builds the nodes for you. On your laptop you need two small helpers: one to turn a Python list into a linked list, and one to turn a linked list back into a Python list so you can print it and check it. Run this cell; the tasks use both.

In [ ]:
def build_list(values):
    head = None
    for v in values[::-1]:
        head = ListNode(v, head)
    return head

def to_list(head):
    values = []
    node = head
    while node:
        values.append(node.val)
        node = node.next
    return values

In [ ]:
h = build_list([4, 5, 6])
print(h.val, h.next.val)
print(to_list(h))
print(build_list([]))
print(to_list(None))

```
4 5
[4, 5, 6]
None
[]
```

- `to_list` is the walk you already know, with `append` inside.
- `build_list` goes through the values backwards (`[6, 5, 4]`), and each new node is coupled to everything built so far, the same way you built the train from the back.
- An empty Python list gives `None`, the empty linked list.

### A stretch: reversing a linked list

Turning `1 -> 2 -> 3` into `3 -> 2 -> 1` is a famous problem (LeetCode 206). You don't make new nodes. You walk the train and turn each coupling around, so it points backwards.

The tricky part: once you turn a node's `next` around, you've lost the way to the rest of the train. So before turning, you save the rest in `nxt`. Three names move along together:

- `prev`: the part already reversed (starts as `None`, nothing),
- `node`: the node you're turning now,
- `nxt`: the rest of the train, saved before you turn.

Run it and watch the reversed part grow.

In [ ]:
def reverse(head):
    prev = None
    node = head
    while node:
        nxt = node.next     # 1. save the rest of the train
        node.next = prev    # 2. turn this coupling around
        prev = node         # 3. the reversed part now starts here
        node = nxt          # 4. step on to the rest
        print("reversed so far:", to_list(prev), "  still to do:", to_list(node))
    return prev

print(to_list(reverse(build_list([1, 2, 3]))))

```
reversed so far: [1]   still to do: [2, 3]
reversed so far: [2, 1]   still to do: [3]
reversed so far: [3, 2, 1]   still to do: []
[3, 2, 1]
```

| Step | `node` | `nxt` (saved) | `node.next` becomes | `prev` after |
| --- | --- | --- | --- | --- |
| 1 | 1 | 2 | `None` | 1 (`1`) |
| 2 | 2 | 3 | 1 | 2 (`2 -> 1`) |
| 3 | 3 | `None` | 2 | 3 (`3 -> 2 -> 1`) |
| check | `None` | | | loop stops, return `prev` |

At the end `node` is `None` and `prev` is the old last node, which is now the head. If this doesn't click yet, that's normal. Draw the three coaches on paper and move the arrows by hand, one step per row of the table.

## Part 2: trees

### A family tree

Think of a family tree drawn upside down, with one person at the top. In a **binary tree**, each person has **at most two** children: a left one and a right one.

- Each person is a **node**.
- The one at the top is the **root**.
- A node's two children are its **left** and **right**. Either can be missing (`None`).
- A node with no children at all is a **leaf**.

LeetCode's definition is the same idea as `ListNode`, with two "next" boxes instead of one:

In [ ]:
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

Three boxes now: `val`, `left` and `right`. `TreeNode(5)` is a node with no children: both `left` and `right` are `None`.

### Building a small tree by hand

```
          4
        /   \
       2     7
      / \
     1   3
```

In [ ]:
root = TreeNode(4)
root.left = TreeNode(2)
root.right = TreeNode(7)
root.left.left = TreeNode(1)
root.left.right = TreeNode(3)

print(root.val)
print(root.left.val)
print(root.left.right.val)
print(root.right.left)

```
4
2
3
None
```

`root.left.right` reads "from the root, go left, then go right": that's the 3. `root.right.left` is `None`, because 7 has no children. 7, 1 and 3 are leaves.

The same tree in one line, with the children passed in as arguments:

```python
TreeNode(4, TreeNode(2, TreeNode(1), TreeNode(3)), TreeNode(7))
```

Read it from the outside in: a 4, whose left is (a 2 with children 1 and 3), and whose right is a 7. On LeetCode, this tree is written `root = [4,2,7,1,3]`.

Run this cell to make some trees for the rest of the notebook:

In [ ]:
#        4
#      /   \
#     2     7
#    / \
#   1   3
small = TreeNode(4, TreeNode(2, TreeNode(1), TreeNode(3)), TreeNode(7))

single = TreeNode(5)          # one node, no children

#   1
#    \
#     2
#      \
#       3
chain = TreeNode(1, None, TreeNode(2, None, TreeNode(3)))

empty = None                  # an empty tree

### Recursion on a tree: each node asks its children

Every child is the root of a **smaller tree**. So a tree problem can be solved with recursion (notebooks 17 and 18): "my answer = my own value, plus whatever my left tree says, plus whatever my right tree says". The base case is the empty tree, `if not node:`.

Here's the sum of a tree, with a `print` so you can watch who answers when. **Predict first:** which node do you think prints first?

In [ ]:
def tree_sum(node):
    if not node:
        return 0
    left = tree_sum(node.left)
    right = tree_sum(node.right)
    total = node.val + left + right
    print(f"at {node.val}: left gives {left}, right gives {right}, so I return {total}")
    return total

print(tree_sum(small))

```
at 1: left gives 0, right gives 0, so I return 1
at 3: left gives 0, right gives 0, so I return 3
at 2: left gives 1, right gives 3, so I return 6
at 7: left gives 0, right gives 0, so I return 7
at 4: left gives 6, right gives 7, so I return 17
17
```

The **leaves answer first**, and the root answers last. That's because 4 can't finish until it has heard from 2 and 7, and 2 can't finish until it has heard from 1 and 3:

```
 tree_sum(4)  waits for left and right
 ├── tree_sum(2)  waits for left and right
 │   ├── tree_sum(1)  -> both children are None, so 1 + 0 + 0 = 1
 │   └── tree_sum(3)  -> 3 + 0 + 0 = 3
 │   so 2 returns 2 + 1 + 3 = 6
 └── tree_sum(7)  -> 7 + 0 + 0 = 7
 so 4 returns 4 + 6 + 7 = 17
```

Every `None` child (and there are six of them here) answers 0 straight away. That's the base case doing its job.

### Height of a tree

Same shape, different question. How many levels does the tree have? A node's height is 1 (itself) plus the height of its **taller** child. An empty tree has height 0.

In [ ]:
def height(node):
    if not node:
        return 0
    answer = 1 + max(height(node.left), height(node.right))
    print(f"height at {node.val} is {answer}")
    return answer

print(height(small))

```
height at 1 is 1
height at 3 is 1
height at 2 is 2
height at 7 is 1
height at 4 is 3
3
```

The 2 has children of height 1, so it's 2. The 7 is a leaf, height 1. The root takes the bigger one, 2, and adds itself: 3.

**Try it:** run `print(height(chain))` and `print(tree_sum(single))` in a new cell. Predict first.

## Common mistakes

**Forgetting `if not node:` in a tree function.** The recursion keeps going down until it reaches a `None` child and asks it for `.val`:

In [ ]:
def broken_sum(node):
    return node.val + broken_sum(node.left) + broken_sum(node.right)

broken_sum(small)

```
AttributeError: 'NoneType' object has no attribute 'val'
```

Every tree function starts with the empty-tree case.

**Forgetting `node = node.next`.** Without it, `node` never changes, `while node:` stays `True`, and the loop never ends. In Jupyter, the cell shows `[*]` and never gets a number. Don't panic: press the **■ stop** button in the toolbar (or **Kernel → Interrupt Kernel**), add the missing line, and run again.

**Walking with `head` itself.** `while head: ... head = head.next` works, but at the end `head` is `None` and you've lost the start of your list. Walk with a separate `node`.

**Running a task cell before the cells above it.** The tasks use `ListNode`, `build_list`, `to_list`, `TreeNode` and the trees from earlier cells. If you see `NameError: name 'build_list' is not defined`, run the cells above first (menu **Run → Run All Above Selected Cell**).

---

# ✏️ Your turn

The tasks use `build_list` and `to_list` so the check lines stay short. A check like `check(to_list(x), [1, 2])` means "turn your linked list into a Python list, and compare".

### Task 1: build one by hand

Build the linked list `7 -> 8 -> 9` with `ListNode`, by hand (not with `build_list`), and call its head `my_head`.

In [ ]:
# write your code below

In [ ]:
from checks import check
check(my_head.val, 7)
check(to_list(my_head), [7, 8, 9])

### Task 2: how long is it?

Write `length(head)` that returns how many nodes the list has.

In [ ]:
def length(head):
    # your code here
    pass

In [ ]:
check(length(build_list([4, 5, 6])), 3)
check(length(build_list([1])), 1)
check(length(None), 0)

### Task 3: is it in there?

Write `contains(head, target)` that returns `True` if some node holds `target`, and `False` if none does.

In [ ]:
def contains(head, target):
    # your code here
    pass

In [ ]:
check(contains(build_list([3, 1, 4]), 1), True)
check(contains(build_list([3, 1, 4]), 9), False)
check(contains(None, 5), False)

### Task 4: the last value

Write `last_value(head)` that returns the value in the last node. For an empty list, return `None`.

Hint: `while node.next:` stops **on** the last node. But think about the empty list first.

In [ ]:
def last_value(head):
    # your code here
    pass

In [ ]:
check(last_value(build_list([4, 5, 6])), 6)
check(last_value(build_list([9])), 9)
check(last_value(None), None)

### Task 5: LeetCode 876, Middle of the Linked List

Return the **middle node** of the list. If there are two middle nodes (an even number of nodes), return the **second** one.

- `[1,2,3,4,5]` → the node holding 3 (and the list from there is `[3,4,5]`)
- `[1,2,3,4,5,6]` → the node holding 4 (`[4,5,6]`)

Hint: count the nodes first. Then start again at the head and take `count // 2` steps.

In [ ]:
class Solution:
    def middleNode(self, head: ListNode | None) -> ListNode | None:
        # your code here
        pass

In [ ]:
check(to_list(Solution().middleNode(build_list([1, 2, 3, 4, 5]))), [3, 4, 5])
check(to_list(Solution().middleNode(build_list([1, 2, 3, 4, 5, 6]))), [4, 5, 6])
check(to_list(Solution().middleNode(build_list([1]))), [1])

### Task 6: count the nodes in a tree

Write `count_nodes(root)` that returns how many nodes the tree has. Use the same shape as `tree_sum`: an empty tree has 0 nodes, otherwise it's 1 (this node) plus the counts on the left and right.

In [ ]:
def count_nodes(root):
    # your code here
    pass

In [ ]:
check(count_nodes(small), 5)
check(count_nodes(single), 1)
check(count_nodes(chain), 3)
check(count_nodes(empty), 0)

### Task 7: LeetCode 104, Maximum Depth of Binary Tree

Return the number of levels in the tree: the number of nodes on the longest path from the root down to a leaf. (It's the `height` from above. Write it again yourself without looking.)

In [ ]:
class Solution:
    def maxDepth(self, root: TreeNode | None) -> int:
        # your code here
        pass

In [ ]:
check(Solution().maxDepth(small), 3)
check(Solution().maxDepth(single), 1)
check(Solution().maxDepth(chain), 3)
check(Solution().maxDepth(empty), 0)

### Task 8 (a stretch): LeetCode 206, Reverse Linked List

Reverse the list and return the new head. Try it without scrolling up. If you get stuck, draw three coaches on paper and use the table from the reversing section.

In [ ]:
class Solution:
    def reverseList(self, head: ListNode | None) -> ListNode | None:
        # your code here
        pass

In [ ]:
check(to_list(Solution().reverseList(build_list([1, 2, 3, 4, 5]))), [5, 4, 3, 2, 1])
check(to_list(Solution().reverseList(build_list([1, 2]))), [2, 1])
check(to_list(Solution().reverseList(None)), [])

---

# ✅ Quiz: check yourself

Put your letter between the quotes, then run the check cell under it.

**Question 1.** `head` is the list `5 -> 8`. What is `head.next.next`?

- A) `8`
- B) `None`
- C) An error

In [ ]:
answer_1 = "?"

In [ ]:
from checks import check_answer
check_answer("20-q1", answer_1)

**Question 2.** After `while node: node = node.next` finishes, what is `node`?

- A) `None`
- B) The last node

In [ ]:
answer_2 = "?"

In [ ]:
check_answer("20-q2", answer_2)

**Question 3.** In `tree_sum`, which line stops the recursion from going on forever?

- A) `return total`
- B) `if not node: return 0`

In [ ]:
answer_3 = "?"

In [ ]:
check_answer("20-q3", answer_3)

**Question 4.** You run a linked list cell, and it shows `[*]` and never finishes. What most likely happened?

- A) Jupyter is saving the notebook
- B) The list is empty
- C) The loop never ends, probably a missing `node = node.next`

In [ ]:
answer_4 = "?"

In [ ]:
check_answer("20-q4", answer_4)

---

## Done

You can now build and walk a linked list without falling off the end, reverse one, build a tree, and solve tree problems by letting each node ask its children.

Stuck on a task? The answers are in `solutions/20-linked-lists-and-trees.ipynb`. Try for real first.

**Next:** better tools for counting, grouping and queues (`21-collections`).